FARK-SIM Agent Initialisation Pipeline

Stratified sampling from Nemotron-Personas-Singapore

Stratified by: gender × age_band × relationship_status

Target: 100 agents (48F, 52M), fitted to M&P 2021 proportions

Step 1: Load Dataset, inspect

In [5]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import json
import uuid

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ─────────────────────────────────────────────
# 1. LOAD AND INSPECT
# ─────────────────────────────────────────────

ds = load_dataset("nvidia/Nemotron-Personas-Singapore")
df = pd.DataFrame(ds["train"])

print("=== Raw Dataset Columns ===")
print(df.columns.tolist())
print(f"\nTotal records: {len(df)}")
print("\nSample row:")
print(df.iloc[0].to_dict())

=== Raw Dataset Columns ===
['uuid', 'professional_persona', 'sports_persona', 'arts_persona', 'travel_persona', 'culinary_persona', 'persona', 'cultural_background', 'skills_and_expertise', 'skills_and_expertise_list', 'hobbies_and_interests', 'hobbies_and_interests_list', 'career_goals_and_ambitions', 'sex', 'age', 'marital_status', 'education_level', 'occupation', 'industry', 'planning_area', 'country']

Total records: 148000

Sample row:
{'uuid': 'd792702ec018494e8e49c69120759408', 'professional_persona': 'Yi Peng Yong, known as Danelle, a 20‑year‑old retail operations associate, leverages her methodical planning, bilingual communication, and data‑driven decision‑making to keep inventory flowing smoothly and nurture harmonious vendor relationships while adhering to a strict routine.', 'sports_persona': 'Yi Peng Yong, known as Danelle, enjoys playing golf at Marina Bay Golf Club, follows the Singapore Open and the PGA Tour, and fits short cardio sessions into her weekly routine to c

2. Normalise Fields 

Adjust field names here if Nemotron uses different names

Common Nemotron field names based on HuggingFace docs

In [6]:
FIELD_MAP = {
    "age":            "age",
    "gender":         "sex",                    # or "gender" — check your print above
    "rel_status":     "marital_status",
    "education":      "education_level",
    "occupation":     "occupation",
    "industry":       "industry",
    "planning_area":  "planning_area",
    "persona":        "general_persona",
    "cultural_bg":    "cultural_background",
    "hobbies":        "hobbies_and_interests",
    "career_goals":   "career_goals_and_ambitions",
}

# Rename to internal names for cleanliness
df = df.rename(columns={v: k for k, v in FIELD_MAP.items()})

cols = [
    "age",
    "gender",
    "rel_status",
    "education",
    "occupation",
    "industry",
    "planning_area",
    "persona",
    "cultural_bg",
    "hobbies",
    "career_goals"
]

df = df[cols].copy()

# Filter age 21–45
df = df[(df["age"] >= 21) & (df["age"] <= 45)].copy()

# Normalise gender
df["gender"] = df["gender"].str.lower().str.strip()
df = df[df["gender"].isin(["male", "female"])].copy()

# Normalise relationship status (dataset has single/married/divorced)
df["rel_status"] = df["rel_status"].str.lower().str.strip()
df = df[df["rel_status"].isin(["single", "married"])].copy()

print(f"\nAfter filtering (age 21-45, valid gender/rel): {len(df)} records")



After filtering (age 21-45, valid gender/rel): 62669 records


In [7]:
TARGETS = {
    ("male",   "single"):  50,
    ("female", "single"):  47,
    ("male",   "married"): 46,
    ("female", "married"): 57,
}
 
assert sum(TARGETS.values()) == 200, "Total must be 200"
assert sum(v for (g,_),v in TARGETS.items() if g == "male")   == 96,  "Male must be 96"
assert sum(v for (g,_),v in TARGETS.items() if g == "female") == 104, "Female must be 104"
 

4. STRATIFIED SAMPLING (age/education preserved as-is from Nemotron)

In [8]:
sampled_frames = []
sampling_log = {}
 
for (gender, rel), n in TARGETS.items():
    cell = df[(df["gender"] == gender) & (df["rel_status"] == rel)]
    available = len(cell)
 
    if available >= n:
        sample = cell.sample(n, random_state=RANDOM_SEED)
        method = "without_replacement"
    else:
        sample = cell.sample(n, replace=True, random_state=RANDOM_SEED)
        method = f"with_replacement (only {available} available)"
        print(f"WARN ({gender}, {rel}): needed {n}, had {available} -- sampled with replacement")
 
    sampling_log[(gender, rel)] = {"needed": n, "available": available, "method": method}
    sampled_frames.append(sample)
 
agents_df = pd.concat(sampled_frames).reset_index(drop=True)

5. ASSIGN IDs + NEUTRAL TPB BASELINE (Step 0)

In [9]:
 
agents_df["agent_id"] = [f"agent_{i:03d}" for i in range(len(agents_df))]

6. POST-HOC PLAUSIBILITY CHECK vs M&P 2021 Annex A

We did NOT fit age or education -- these are whatever Nemotron gave us. This block just reports them so you can note alignment/divergence.

In [12]:
print("\n=== Post-Hoc Check vs M&P 2021 Annex A ===")
print(f"\nTotal agents: {len(agents_df)}  (expect 200)")
 
print("\nGender x Marital (stratified -- should match targets exactly):")
print(pd.crosstab(agents_df["gender"], agents_df["rel_status"]).to_string())
 
print("\nRelationship status (after dating subdivision):")
print(agents_df["rel_status"].value_counts().to_string())
 
# Age -- M&P targets shown for reference (NOT fitted)
print("\nAge band -- SINGLE  [M&P ref: 21-25=30%, 26-30=34%, 31-35=18%, 36-40=10%, 41-45=9%]")
single = agents_df[agents_df["rel_status"] == "single"]
print((pd.cut(single["age"], [20,25,30,35,40,45],
              labels=["21-25","26-30","31-35","36-40","41-45"])
       .value_counts(normalize=True).sort_index()*100).round(1).to_string())
 
print("\nAge band -- MARRIED [M&P ref: 21-25=1%, 26-30=9%, 31-35=24%, 36-40=30%, 41-45=37%]")
married = agents_df[agents_df["rel_status"] == "married"]
print((pd.cut(married["age"], [20,25,30,35,40,45],
              labels=["21-25","26-30","31-35","36-40","41-45"])
       .value_counts(normalize=True).sort_index()*100).round(1).to_string())
 
# Education -- M&P targets shown for reference (NOT fitted)
print("\nEducation -- SINGLE  [M&P ref: Secondary<=12%, Diploma/A=34%, Degree+=55%]")
print(single["education"].value_counts(normalize=True).mul(100).round(1).to_string())
 
print("\nEducation -- MARRIED [M&P ref: Secondary<=16%, Diploma/A=28%, Degree+=56%]")
print(married["education"].value_counts(normalize=True).mul(100).round(1).to_string())


=== Post-Hoc Check vs M&P 2021 Annex A ===

Total agents: 200  (expect 200)

Gender x Marital (stratified -- should match targets exactly):
rel_status  married  single
gender                     
female           57      47
male             46      50

Relationship status (after dating subdivision):
rel_status
married    103
single      97

Age band -- SINGLE  [M&P ref: 21-25=30%, 26-30=34%, 31-35=18%, 36-40=10%, 41-45=9%]
age
21-25    35.1
26-30    33.0
31-35    20.6
36-40     7.2
41-45     4.1

Age band -- MARRIED [M&P ref: 21-25=1%, 26-30=9%, 31-35=24%, 36-40=30%, 41-45=37%]
age
21-25     2.9
26-30     9.7
31-35    21.4
36-40    30.1
41-45    35.9

Education -- SINGLE  [M&P ref: Secondary<=12%, Diploma/A=34%, Degree+=55%]
education
University                       52.6
Post Secondary (Non-Tertiary)    14.4
Polytechnic                      11.3
Other Diploma                    11.3
Secondary                         7.2
Lower Secondary                   2.1
Primary                   

7. Export to csv

In [17]:
import json

def row_to_agent(row):
    return {
        "agent_id":               row["agent_id"],
        "age":                    int(row["age"]),
        "gender":                 row["gender"],
        "rel_status":             row["rel_status"],
        "education":              row.get("education", None),
        "occupation":             row.get("occupation", None),
        "industry":               row.get("industry", None),
        "planning_area":          row.get("planning_area", None),
        "general_persona":        row.get("persona", None),
        "cultural_background":    row.get("cultural_bg", None),
        "hobbies_and_interests":  row.get("hobbies", None),
        "career_goals":           row.get("career_goals", None),
 
        # TPB belief state -- neutral baseline (updated after seed memories)
        "belief_state": {
            "attitude_score":           3,
            "subjective_norm_score":    3,
            "pbc_score":                3,
            "fertility_intention_dist": None,
        },
 
        # Memory stream -- populated in next pipeline step
        "memory_stream": [],
    }
 
agents = [row_to_agent(row) for _, row in agents_df.iterrows()]
 
with open("agents_initialised.json", "w", encoding="utf-8") as f:
    json.dump(agents, f, indent=2, ensure_ascii=False)
 
print(f"\nSaved {len(agents)} agents to agents_initialised.json")
print("Next step: generate financial security scores + seed memories per agent")
 


Saved 200 agents to agents_initialised.json
Next step: generate financial security scores + seed memories per agent
